# 14 — Advanced Testing: Properties, Integration, Fakes, and CI

Reliable tests focus on behavioral contracts, invariants, independent fixtures, and realistic boundaries. Use stdlib unittest for executable examples; learn how property-based thinking extends beyond fixed examples.

*Advanced · 90–150 min*

## You will learn

- Write invariants and table-driven tests
- Use deterministic random generation for test cases
- Test adapters with temporary databases and fakes
- Explain a layered testing strategy

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Property-style reasoning

A property describes behavior across many valid inputs. Rather than proving correctness with a single example, generate reproducible cases and assert invariants. External libraries like Hypothesis can automate shrinking, but aren't required here.

In [ ]:
import random

def reverse_twice(values):
    return list(reversed(list(reversed(values))))

rng = random.Random(42)
for _ in range(30):
    values = [rng.randint(-100, 100) for _ in range(rng.randint(0, 20))]
    assert reverse_twice(values) == values
print("30 generated cases passed")

**Design review:** What property would catch a sorting implementation that loses duplicates?

### 2. Table-driven boundaries

Table-driven tests collect expected values and edge cases. unittest.subTest reports which case fails. Focus on empty, negative, very large, and invalid inputs—not only the happy path.

In [ ]:
import unittest

def clamp(number, low, high):
    if low > high:
        raise ValueError("invalid range")
    return min(max(number, low), high)

class ClampTests(unittest.TestCase):
    def test_cases(self):
        for value, expected in [(-1, 0), (4, 4), (20, 10)]:
            with self.subTest(value=value):
                self.assertEqual(clamp(value, 0, 10), expected)

result = unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(ClampTests))
assert result.wasSuccessful()

**Design review:** What changes when a test uses random inputs without recording the seed?

### 3. Integration tests and temporary storage

Integration tests exercise multiple real components together, e.g. a repository and SQLite connection. In-memory databases avoid persistent side effects. Tests should still include transaction and constraint behavior.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE items(id INTEGER PRIMARY KEY, code TEXT UNIQUE)")
    db.execute("INSERT INTO items(code) VALUES (?)", ("A",))
    try:
        db.execute("INSERT INTO items(code) VALUES (?)", ("A",))
    except sqlite3.IntegrityError:
        print("Duplicate rejected")
    db.rollback()
    assert db.execute("SELECT COUNT(*) FROM items").fetchone()[0] == 0
finally:
    db.close()

**Design review:** Why did rollback remove the first insert too, and how would an earlier commit change that?

### 4. Mocks versus behavior

Mocks are appropriate at external boundaries (network, email, time). Avoid over-mocking internal helper functions, which makes tests fragile during refactoring. Check outcomes and meaningful interactions.

In [ ]:
from unittest.mock import Mock

def notify(service, message):
    return service.send(message)

fake = Mock()
fake.send.return_value = "queued"
assert notify(fake, "ready") == "queued"
fake.send.assert_called_once_with("ready")
print("Fake transport verified")

**Design review:** Why is a fake useful for simulating network failure deterministically?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/03_security_testing_performance_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/14', {1: '3fd3b89960bbd2a8', 2: '34ff1a6cdd7cb5c5', 3: '3b8be8cd7f47b95f', 4: 'cfeea47a710c7ed0'})

**Exercise 1.** An invariant test: sorting keeps the same elements and puts them in order.

In [ ]:
import random
rng = random.Random(1)
ok = True
for _ in range(200):
    values = [rng.randint(-50, 50) for _ in range(rng.randint(0, 20))]
    result = sorted(values)
    ok &= sorted(result) == result and ___ and len(result) == len(values)
check(1, ok)

**Exercise 2.** Seeded random test data is reproducible.

In [ ]:
import random
first = [random.Random(___).randint(1, 100) for _ in range(3)]
second = [random.Random(123).randint(1, 100) for _ in range(3)]
check(2, first == second)

**Exercise 3.** An integration test against a temporary SQLite database.

In [ ]:
import sqlite3, tempfile, os

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "test.db")
    with sqlite3.connect(path) as conn:
        conn.execute("CREATE TABLE notes (text TEXT)")
        conn.execute("INSERT INTO notes VALUES (?)", (___,))
    with sqlite3.connect(path) as conn:
        rows = conn.execute("SELECT text FROM notes").fetchall()
    conn.close()
check(3, rows)

**Exercise 4.** A fake replaces the slow dependency in a unit test.

In [ ]:
class FakeMailer:
    def __init__(self): self.sent = []
    def send(self, to, body): self.sent.append((to, body))

def welcome(mailer, user):
    mailer.send(user, f"Welcome, {user}!")

mailer = ___
welcome(mailer, "ana")
check(4, mailer.sent)

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** A property-based test checks…

   - **a)** An invariant across many generated inputs
   - **b)** Only performance
   - **c)** One example

**2.** Why seed random test data?

   - **a)** Faster tests
   - **b)** Failures become reproducible
   - **c)** More coverage

**3.** Integration tests against a temporary SQLite DB check…

   - **a)** Only pure functions
   - **b)** The network
   - **c)** Your code with a real adapter

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/14', {1: 'feb2d351bd74', 2: 'b5ed09fcd6ed', 3: 'e36e6d620f68'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [15 — Observability, Structured Logging, Metrics, and SLOs](15_observability_and_slos.ipynb)